In [ ]:
!pip install torch tiktoken datasets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

In [ ]:
@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

class SimpleTokenizer:
    def __init__(self, config=None):
        self.config = config or TokenizerConfig()
        self.enc = tiktoken.get_encoding(self.config.name)
        self.eos_token = "<|endoftext|>"
        self.eos_token_id = self.enc.encode(self.eos_token, allowed_special={self.eos_token})[0]

    def encode(self, text):
        return self.enc.encode(text, allowed_special={self.eos_token})

    def decode(self, ids):
        return self.enc.decode(ids)


In [ ]:

# (We include the minimal blocks needed to load the weights)
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = freqs.repeat_interleave(2, dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

def create_causal_mask(seq_len, device):
    return torch.tril(torch.ones(seq_len, seq_len, device=device)).view(1, 1, seq_len, seq_len)


In [ ]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv_proj(x).reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = self.rotary(q, seq_len), self.rotary(k, seq_len)
        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = self.attn_dropout(F.softmax(attn_scores, dim=-1))
        attn_output = (attn_weights @ v).transpose(1, 2).contiguous().reshape(batch_size, seq_len, self.d_model)
        return self.resid_dropout(self.out_proj(attn_output))

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    d_model: int = 768
    num_heads: int = 12
    num_layers: int = 12
    max_seq_len: int = 512
    dropout: float = 0.1
    embd_dropout: float = 0.1

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.embd_dropout = nn.Dropout(config.embd_dropout)
        self.layers = nn.ModuleList([TransformerBlock(config.d_model, config.num_heads, config.dropout) for _ in range(config.num_layers)])
        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        x = self.embd_dropout(self.token_embedding(input_ids))
        mask = create_causal_mask(seq_len, input_ids.device)
        for layer in self.layers: x = layer(x, mask)
        return self.lm_head(self.final_norm(x))

In [ ]:
class PIIDataset(Dataset):
    def __init__(self, data, tokenizer, max_seq_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # 1. Format the strings
        raw_prompt = f"[RAW] {item['original_text']} [REDACTED] "
        redacted_answer = f"{item['redacted_text']}{self.tokenizer.eos_token}"

        # 2. Tokenize separately to calculate lengths
        prompt_ids = self.tokenizer.encode(raw_prompt)
        answer_ids = self.tokenizer.encode(redacted_answer)

        # 3. Combine them
        full_ids = prompt_ids + answer_ids

        # 4. Truncate if too long
        if len(full_ids) > self.max_seq_len + 1:
            full_ids = full_ids[:self.max_seq_len + 1]

        # 5. Create Targets & Apply LOSS MASKING (-100)
        # We set the target of the prompt portion to -100 so the model doesn't calculate loss on it
        targets = full_ids[1:] + [self.tokenizer.eos_token_id]

        target_tensor = torch.tensor(targets, dtype=torch.long)
        # Mask out the prompt length!
        prompt_length = len(prompt_ids)
        target_tensor[:prompt_length-1] = -100

        input_tensor = torch.tensor(full_ids, dtype=torch.long)

        # Pad to max_seq_len to keep batches uniform
        pad_len = self.max_seq_len - len(input_tensor)
        if pad_len > 0:
            input_tensor = F.pad(input_tensor, (0, pad_len), value=self.tokenizer.eos_token_id)
            target_tensor = F.pad(target_tensor, (0, pad_len), value=-100)

        return input_tensor, target_tensor

In [ ]:

print("Initializing PII Fine-Tuning...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = SimpleTokenizer()

# Load your existing Base Model!
print("Loading pre-trained model.pt...")
checkpoint = torch.load("model.pt", map_location=device, weights_only=False)
model = GPT(checkpoint['config'])
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

# We lower the learning rate for fine-tuning! We don't want to destroy the base knowledge.
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

print("Loading AI4Privacy PII dataset...")
# Using a great open-source PII masking dataset
dataset = load_dataset("ai4privacy/pii-masking-200k", split="train") # Taking 5k examples for quick tuning

# We now know the exact column names!
# 'source_text' is the raw text, and 'target_text' is the clean inline redacted text.
print(f"Dataset columns found: {dataset.column_names}")
print("Using 'target_text' as the redacted text column.")

# Rename columns to match our dataset class
formatted_data = [{"original_text": row["source_text"], "redacted_text": row["target_text"]} for row in dataset]

train_dataset = PIIDataset(formatted_data, tokenizer, max_seq_len=256)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print("\nStarting Supervised Fine-Tuning (SFT)...")
model.train()
epochs = 2

for epoch in range(epochs):
    for step, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(inputs)

        # The -100 in targets makes CrossEntropy ignore the [RAW] prompt!
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)

        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.4f}")

    print("Fine-tuning complete! Saving pii_model.pt...")
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'config': model.config
    }
    torch.save(checkpoint, 'pii_model.pt')
    print("Saved successfully!")